## importing requre library


In [0]:
from pyspark.sql import functions as F  
from delta.tables import DeltaTable 

In [0]:
%run "/Workspace/Users/abdulm63633@gmail.com/Ecommerce Lakehouse Project/01_setup_file/setup_utils"

In [0]:
print(bronze_schema,silver_schema,gold_schema)

In [0]:
dbutils.widgets.text("catalog", "ecommerce_lakehouse_project", "Catalog")
dbutils.widgets.text("data_source", "products", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")



In [0]:
df_bronze = spark.sql(f"SELECT * FROM {catalog}.{bronze_schema}.{data_source};")
df_bronze.show()

In [0]:
# ================================= QUALITY CHECK ======================

# 01 checking category columm
df_bronze.select("category").distinct().show()

# Checking null count in each column 
null_counts = df_bronze.select([F.count(F.when(F.isnull(c), c)).alias(c) for c in df_bronze.columns])
null_counts.show()
## checking brand column
df_bronze.select("brand").distinct().show()

## checking create_at column

df_bronze.groupBy("product_id","product_name","category","brand","created_at").count().filter(F.col("count") > 1).show()

# check price < 0  or null valus
df_bronze.filter((F.col("price") < 0) | (F.col("price").isNull()) ).show()

In [0]:

# Remove exact duplicate records
df_silver = df_bronze.dropDuplicates(["product_id","product_name","category","brand","created_at"])
df_silver.show()

In [0]:
# remove black_space and capitalize name
df_silver = df_silver.withColumn("product_name",
            F.trim(F.initcap("product_name"))                                 
)

df_silver.show()


In [0]:
df_valid_price =  ((F.col("price") > 0) &
    (F.col("price").isNotNull()))

df_valid = df_silver.filter(df_valid_price)
df_invalid =  df_silver.filter(~df_valid_price)

df_silver = df_valid

df_invalid.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(
        f"{catalog}.quarantine.bad_product_price"
    )

In [0]:
df_silver = df_silver.dropDuplicates(["product_id"])

if not (spark.catalog.tableExists(f"{catalog}.{silver_schema}.{data_source}")):
    df_silver.write \
    .format("delta") \
    .option("delta.enableChangeDataFeed", "true") \
    .mode("overwrite") \
    .saveAsTable(
        f"{catalog}.{silver_schema}.{data_source}"
    )
    print("sucessfully writed  data to delta location")
else:
    print("Doing upsert operation")
    delta_table = DeltaTable.forName(
        spark,
        f"{catalog}.{silver_schema}.{data_source}"
    )

    delta_table.alias("target").merge(
        source=df_silver.alias("source"),
        condition="""
            target.product_id = source.product_id
        """
    ).whenMatchedUpdate(
        condition="""
        NOT (target.product_name <=> source.product_name)
        OR NOT (target.category <=> source.category)
        OR NOT (target.brand <=> source.brand)
        OR NOT (target.price <=> source.price)
        OR NOT (target.created_at <=> source.created_at)
    """,
        set={
             "product_name": "coalesce(source.product_name, target.product_name)",
            "category": "coalesce(source.category, target.category)",
            "brand": "coalesce(source.brand, target.brand)",
            "price": "coalesce(source.price, target.price)",
            "created_at": "coalesce(source.created_at, target.created_at)"
            
        }
    ).whenNotMatchedInsert(
        values={
            "product_id": "source.product_id",
            "product_name": "source.product_name",
            "category": "source.category",
            "brand": "source.brand",
            "price": "source.price",
            "created_at":"source.created_at"
        }
    ).execute()

